# BCAL Jupyter Demo

A minimal, self-contained walk-through of the BCAL audit pipeline. Uses
a small synthetic dataset — no external data or API keys required.

```
pip install -e .[dev]
```

In [ ]:
import numpy as np
import pandas as pd

from bcal import Instrument
from bcal.modules import batch_audit, outlier_ledger, subgroup, version_lock
from bcal.profiles import load_profile
from bcal.report import write_json, write_markdown

rng = np.random.default_rng(seed=0)

## 1. Build a synthetic sample-level dataset

In [ ]:
n = 120
df = pd.DataFrame({
    'sample_id':   [f'S-{i:03d}' for i in range(n)],
    'treatment':   rng.integers(0, 2, size=n),
    'outcome':     rng.normal(size=n),
    'sex':         rng.choice(['M', 'F'], size=n),
    'age_bucket':  rng.choice(['<50', '50-65', '>65'], size=n),
    'batch':       rng.choice(['Batch_1', 'Batch_2'], size=n),
    'subtype':     rng.choice(['Resistant', 'Sensitive'], size=n),
}).set_index('sample_id')
df.head()

## 2. Run the five audit modules

In [ ]:
# M1 — subgroup consistency
sub_findings = subgroup.run(
    df, treatment_col='treatment', outcome_col='outcome',
    covariate_cols=['sex', 'age_bucket']
)
for f in sub_findings:
    print(f.subgroup_id, f.status, round(f.effect, 3))

In [ ]:
# M4 — batch audit on a fake feature matrix
features = rng.normal(size=(n, 200)) + (df['batch'] == 'Batch_2').to_numpy()[:, None]
audit = batch_audit.run(features, df, batch_col='batch', subtype_col='subtype')
print('PVCA variance attributable to batch:', audit.variance_ratio)

In [ ]:
# M5 — outlier classification
depth = pd.Series(rng.normal(loc=10_000_000, scale=1_000_000, size=n), index=df.index)
depth.iloc[0] = 100_000  # planted outlier
flagged = outlier_ledger.screen(depth, threshold=3.0)
records = outlier_ledger.classify(flagged, evidence_map={'S-000': 'low library complexity'})
records

## 3. Seal into a SEP and emit reports

In [ ]:
prof = load_profile('fda_ind')

with Instrument(pipeline_name='demo', version='0.1.0', operator='notebook') as inst:
    inst.set_manifest_digest('0' * 64)  # placeholder — real runs use hash_inputs()
    inst.add_subgroup_findings(sub_findings)
    inst.set_batch_audit(audit)
    inst.add_outliers(records)
    inst.add_checklist_items(prof.checklist)
    inst.capture_tool_version('numpy')
    sep = inst.seal()

write_json(sep, 'demo-sep.json')
write_markdown(sep, 'demo-sep.md')
print('Seal:', sep.seal[:16] + '…')